# Component_01 — STAGE 4: Report Generator Rebuild
### ConvNeXt-Base vision → BART/BioBART encoder-decoder

This is **Model 2** — image in, radiology report out.

---

## Why the old one failed (measured, not guessed)

From 100 generations of your previous model:

| | |
|---|---|
| Unique opening sentences | **14 / 100** |
| One output emitted verbatim | **32 times** |
| Vocabulary | 248 words vs 1,046 in references (**ratio 0.24**) |
| ROUGE-L | 0.2739 — **below** the 0.2769 constant baseline |

A fixed string that ignores the X-ray entirely scores **0.2769**. The old model
scored **0.2739**. It learned nothing from the image.

## Root cause — one line

```python
self.bart(encoder_outputs=BaseModelOutput(last_hidden_state=vis), ...)   # OLD
```

This **bypasses BART's pretrained encoder entirely**. The decoder's cross-attention
was pretrained to read encoder outputs with a specific scale, geometry and
positional structure. It received raw projected ConvNeXt features instead, could
not use them, and fell back on its language prior. Compounding it:

- **no positional encoding** on the 144 visual tokens — apex and base are indistinguishable
- the only trainable image path was one `Linear(1024→768)`
- beam search with `early_stopping=True, length_penalty=1.0` drives output toward the corpus-modal string

## The fix

```python
self.bart(inputs_embeds=projected_visual_tokens, labels=labels)          # NEW
```

Passing `inputs_embeds` sends the visual tokens **through** BART's encoder, and
BART adds its own learned positional embeddings on the way in. One change repairs
both defects at once — the encoder is used, and position is restored.

---

## Every Stage-5 bug, already fixed here

| Stage-5 bug | Fixed in this notebook |
|---|---|
| `deepcopy(model)` for EMA eval → OOM | weight **swap**, backup on CPU |
| eval batch `×2` → OOM | eval batch **=** train batch |
| batch probe stepped with `lr=1e-6` | **`lr=0.0`** |
| `CU_PER_HR["L4"] = 4.8` (2.6× wrong) | **1.75**, measured from your own run |
| `EMA_DECAY=0.9998` too slow for 22k steps | **derived from run length** |
| 1.4 GB Drive write every 10 min | mid-epoch saves go to **local disk** |
| `NUM_WORKERS` hardcoded | from `os.cpu_count()` |
| noisy checkpoint selection | **fixed** val subset → paired comparisons |

Measured target lengths (4,000 real reports, BART tokenizer): median **76** tokens,
p95 ≈ 160, p99 **204**, max 402. `MAX_TOKENS=256` truncates ~0.3%; dynamic padding
to the batch maximum removes ~60% of decoder compute versus padding to the cap.

---
# 0 · Configuration

In [ ]:
CFG = dict(
    # ---- run control -------------------------------------------------------
    SMOKE_TEST        = True,     # ⚠️ leave True first. Then set False.
    RESUME            = True,

    # ---- data --------------------------------------------------------------
    IMG_SIZE          = 384,
    MAX_TOKENS        = 256,      # measured p99 = 204 BPE tokens; 256 truncates ~0.3%
                                  # (old code used 512 with pad-to-max — 4x wasted compute)
    NUM_WORKERS       = 4,

    # ---- model -------------------------------------------------------------
    DECODER           = "GanjinZero/biobart-v2-base",   # falls back to facebook/bart-base
    UNFREEZE_VISION_STAGES = 1,   # last ConvNeXt stage only, at 0.1x LR. 0 = fully frozen
    PROJ_DROPOUT      = 0.1,

    # ---- optimisation ------------------------------------------------------
    EPOCHS            = 15,
    EFFECTIVE_BATCH   = 32,
    BASE_LR           = 5e-5,     # decoder
    PROJ_LR_MULT      = 10.0,     # projection needs to move faster
    VISION_LR_MULT    = 0.1,      # unfrozen vision stage moves slower
    WEIGHT_DECAY      = 0.01,
    WARMUP_EPOCHS     = 1,
    LABEL_SMOOTH      = 0.1,      # standard for seq2seq; also fights mode collapse
    GRAD_CLIP         = 1.0,
    PATIENCE          = 5,

    # ---- evaluation --------------------------------------------------------
    VAL_SUBSET        = 500,      # FIXED subset -> paired epoch-to-epoch comparison
    GEN_MAX_TOKENS    = 192,      # covers ~97% of references (p95 ≈ 160)
    GEN_MIN_TOKENS    = 24,       # stops stubs. MEASURED: min_length=40 would force
                                  # over-generation on 5.3% of references; 24 -> 0.7%
    VAL_BEAMS         = 1,        # greedy during training (fast)
    TEST_BEAMS        = 4,
    LENGTH_PENALTY    = 1.2,      # >1 favours longer -> counteracts collapse
    NO_REPEAT_NGRAM   = 3,

    # ---- regularisation ----------------------------------------------------
    USE_EMA           = True,

    # ---- speed -------------------------------------------------------------
    AMP_DTYPE         = "bf16",
    SEED              = 42,
)
CONST_BASELINE_ROUGEL = 0.2769    # from Stage 1, on the cleaned test set
print("SMOKE_TEST =", CFG["SMOKE_TEST"], "  <-- must be False for the real run")

---
# 1 · Environment

In [ ]:
import os, sys, json, math, time, random, subprocess, re, gc, warnings
warnings.filterwarnings("ignore")
from pathlib import Path
from datetime import datetime
import numpy as np

print("=" * 80); print("  COMPONENT_01 · STAGE 4 · REPORT GENERATOR"); print("=" * 80)
try:
    smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=15)
    GPU = smi.stdout.strip()
except Exception:
    GPU = ""
if not GPU:
    raise SystemExit("No GPU. Runtime → Change runtime type → L4, then re-run.")
print(f"  GPU: {GPU}")
GPU_NAME = GPU.split(",")[0].strip()

# MEASURED on your Stage 5 run: 8.01 CU for ~4.7 h connected = ~1.7 CU/h on L4.
# The old 4.8 figure over-reported every cost estimate by 2.6x.
CU_PER_HR = {"L4": 1.75, "A100": 11.0, "T4": 1.76, "V100": 4.9}
RATE = next((v for k, v in CU_PER_HR.items() if k in GPU_NAME), 2.0)
print(f"  burn rate: ~{RATE} CU/hour  (measured from your Stage 5 run)")
if "T4" in GPU_NAME:
    CFG["AMP_DTYPE"] = "fp16"; print("  ⚠️ T4 → fp16")

need = [p for m, p in [("transformers", "transformers"), ("rouge_score", "rouge-score")]
        if __import__("importlib").util.find_spec(m) is None]
if need:
    print(f"  installing {need} ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)

import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import pandas as pd
from transformers import AutoTokenizer, BartForConditionalGeneration
from rouge_score import rouge_scorer

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.benchmark = True
seed_all(CFG["SEED"])
CFG["NUM_WORKERS"] = max(2, min(CFG["NUM_WORKERS"], os.cpu_count() or 4))
DEV = torch.device("cuda")
AMP_DT = torch.bfloat16 if CFG["AMP_DTYPE"] == "bf16" else torch.float16
print(f"  torch {torch.__version__} | VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB "
      f"| workers {CFG['NUM_WORKERS']}")
T_START = time.time()

---
# 2 · Drive, images, manifests

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

PROJECT  = Path("/content/drive/MyDrive/Component_01")
TAR      = PROJECT / "data" / "images" / "cardio_384.tar"
MANIFEST = PROJECT / "training_manifest"
S5_CKPT  = PROJECT / "checkpoints" / "stage5" / "backbone_for_stage4.pt"
CKPT_DIR = PROJECT / "checkpoints" / "stage4"
REPORTS  = PROJECT / "reports" / "stage4"
IMG_ROOT = Path("/content/cardio_image_384")
for d in (CKPT_DIR, REPORTS): d.mkdir(parents=True, exist_ok=True)

if not IMG_ROOT.exists():
    if not TAR.exists(): raise FileNotFoundError(f"{TAR} missing")
    print(f"extracting {TAR.stat().st_size/1e9:.1f} GB → /content/ ...")
    t0 = time.time(); subprocess.run(["tar", "-xf", str(TAR), "-C", "/content/"], check=True)
    print(f"  {time.time()-t0:.0f}s")
else:
    print("images already on local disk")
n_png = sum(1 for _ in IMG_ROOT.rglob("*.png"))
assert n_png >= 45000, f"only {n_png} PNGs found"
print(f"  {n_png:,} images")

if not S5_CKPT.exists():
    raise FileNotFoundError(f"{S5_CKPT} missing — run Stage 5 first")
print(f"  Stage-5 backbone: {S5_CKPT.stat().st_size/1e6:.0f} MB")

DF = {s: pd.read_csv(MANIFEST / f"manifest_{s}.csv", low_memory=False)
      for s in ("train", "val", "test")}
for s, d in DF.items():
    assert d["dicom_id"].is_unique and d["report"].notna().all()
    print(f"  {s:<6}{len(d):>7,} rows")
sub = {s: set(d.subject_id) for s, d in DF.items()}
assert not (sub["train"] & sub["val"]) and not (sub["train"] & sub["test"])
print("  ✅ zero patient leakage")

---
# 3 · Tokenizer & targets

In [ ]:
try:
    TOK = AutoTokenizer.from_pretrained(CFG["DECODER"])
    print(f"  tokenizer: {CFG['DECODER']}")
except Exception as e:
    print(f"  ⚠️ {CFG['DECODER']} unavailable ({type(e).__name__}) → facebook/bart-base")
    CFG["DECODER"] = "facebook/bart-base"
    TOK = AutoTokenizer.from_pretrained(CFG["DECODER"])

_l = [len(TOK(r, truncation=False)["input_ids"]) for r in DF["train"]["report"].sample(2000, random_state=0)]
_l = np.array(_l)
print(f"  target tokens: mean={_l.mean():.0f} median={np.median(_l):.0f} "
      f"p95={np.percentile(_l,95):.0f} p99={np.percentile(_l,99):.0f} max={_l.max()}")
print(f"  MAX_TOKENS={CFG['MAX_TOKENS']} → truncates {(_l > CFG['MAX_TOKENS']).mean()*100:.2f}% of targets")
print(f"  dynamic padding saves ≈{(1-np.median(_l)/CFG['MAX_TOKENS'])*100:.0f}% of decoder compute vs pad-to-max")

---
# 4 · Transforms, dataset, dynamic-padding collator

In [ ]:
_EPS = 1e-6
class ToGrayscalePIL:
    def __call__(self, img): return img if img.mode == "L" else img.convert("L")
class PerImageZScore:
    def __init__(self, c=3): self.c = c
    def __call__(self, t):
        if t.shape[0] != 1: t = t[:1]
        s = t.std(); t = (t - t.mean())/s if s > _EPS else t - t.mean()
        return t.repeat(self.c, 1, 1)

def build_transform(split, img_size=CFG["IMG_SIZE"]):
    ops = [ToGrayscalePIL(), transforms.Resize((img_size, img_size))]
    if split == "train":
        ops.append(transforms.RandomAffine(degrees=5.0, translate=(0.03, 0.03), scale=(0.97, 1.03),
                   interpolation=transforms.InterpolationMode.BILINEAR, fill=0))
    return transforms.Compose(ops + [transforms.ToTensor(), PerImageZScore(3)])

class CXRReportDataset(Dataset):
    def __init__(self, df, split):
        self.paths = [str(IMG_ROOT / p) for p in df["image_path"]]
        self.reports = df["report"].astype(str).tolist()
        self.tf = build_transform(split)
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = self.tf(Image.open(self.paths[i]))
        ids = TOK(self.reports[i], truncation=True, max_length=CFG["MAX_TOKENS"])["input_ids"]
        return img, torch.tensor(ids, dtype=torch.long), self.reports[i]

def collate(batch):
    """Pad to the longest sequence IN THIS BATCH, not to MAX_TOKENS.
    Measured: median target is 62 tokens vs a 192 cap, so this removes ~67%
    of decoder compute — the single biggest speed win in this notebook."""
    imgs, seqs, texts = zip(*batch)
    n = max(len(s) for s in seqs)
    labels = torch.full((len(seqs), n), -100, dtype=torch.long)
    for i, s in enumerate(seqs):
        labels[i, :len(s)] = s
    return torch.stack(imgs), labels, list(texts)

DS = {s: CXRReportDataset(DF[s], "train" if s == "train" else "eval") for s in DF}

# FIXED validation subset: the same images every epoch, so epoch-to-epoch ROUGE
# comparisons are paired. Random subsets add noise that swamps real improvement —
# the mistake that made Stage 5's early numbers unreadable.
_rng = np.random.RandomState(CFG["SEED"])
VAL_IDX = sorted(_rng.choice(len(DS["val"]), min(CFG["VAL_SUBSET"], len(DS["val"])), replace=False).tolist())
VAL_SUB = torch.utils.data.Subset(DS["val"], VAL_IDX)
print(f"  datasets: " + " | ".join(f"{s}={len(d):,}" for s, d in DS.items()))
print(f"  fixed val subset for ROUGE: {len(VAL_SUB)}")

---
# 5 · Model — the corrected architecture

In [ ]:
class CXRReportGenerator(nn.Module):
    """
    image → ConvNeXt features (B,1024,12,12)
          → flatten to 144 tokens → MLP projection → (B,144,d_model)
          → BART ENCODER  ← the fix: visual tokens go THROUGH the encoder,
                             and BART adds its own learned positional embeddings
          → BART decoder cross-attends → report tokens
    """
    def __init__(self, backbone_path, decoder_name, unfreeze_stages=1):
        super().__init__()
        base = models.convnext_base(weights=None)
        self.vision = base.features
        ck = torch.load(backbone_path, map_location="cpu", weights_only=False)
        feat = ck["features"] if "features" in ck else ck
        feat = {k.replace("features.", ""): v for k, v in feat.items()}
        missing, unexpected = self.vision.load_state_dict(feat, strict=False)
        assert not unexpected, f"unexpected keys in backbone: {list(unexpected)[:5]}"
        print(f"  vision loaded from Stage 5 ({len(feat)} tensors, {len(missing)} missing)")

        for p in self.vision.parameters(): p.requires_grad = False
        self.unfrozen = []
        if unfreeze_stages > 0:
            for m in list(self.vision.children())[-unfreeze_stages * 2:]:
                for p in m.parameters(): p.requires_grad = True
                self.unfrozen.append(m)
            print(f"  unfroze last {unfreeze_stages} ConvNeXt stage(s)")

        self.bart = BartForConditionalGeneration.from_pretrained(decoder_name)
        d = self.bart.config.d_model
        self.proj = nn.Sequential(
            nn.LayerNorm(1024), nn.Linear(1024, d), nn.GELU(),
            nn.Dropout(CFG["PROJ_DROPOUT"]), nn.Linear(d, d), nn.LayerNorm(d))

    def encode(self, images):
        if self.unfrozen:
            f = self.vision(images)
        else:
            with torch.no_grad():
                f = self.vision(images)
        f = f.flatten(2).transpose(1, 2)          # (B,144,1024)
        return self.proj(f)                        # (B,144,d_model)

    def forward(self, images, labels):
        emb = self.encode(images)
        mask = torch.ones(emb.shape[:2], dtype=torch.long, device=emb.device)
        # inputs_embeds → BART's own encoder runs, and adds learned positions.
        return self.bart(inputs_embeds=emb, attention_mask=mask, labels=labels)

    @torch.no_grad()
    def generate(self, images, beams, max_new, min_new):
        emb = self.encode(images)
        mask = torch.ones(emb.shape[:2], dtype=torch.long, device=emb.device)
        # ⚠️ READ THIS BEFORE "FIXING" IT.
        # We run BART's encoder OURSELVES and hand generate() its OUTPUT.
        # This is NOT the old bug. The old code passed RAW projected ConvNeXt
        # features as encoder_outputs, so the encoder never ran. Here the encoder
        # HAS run (line above) and we pass its result — which is exactly the
        # tensor the decoder's cross-attention was pretrained to consume.
        # Verified: identical output to generate(inputs_embeds=...), but it does
        # not depend on the transformers version handling inputs_embeds inside
        # generate(), so it is portable across Colab's transformers builds.
        enc = self.bart.model.encoder(inputs_embeds=emb, attention_mask=mask)
        kw = dict(encoder_outputs=enc, attention_mask=mask, num_beams=beams,
                  max_length=max_new, min_length=min_new,
                  no_repeat_ngram_size=CFG["NO_REPEAT_NGRAM"])
        if beams > 1:
            # length_penalty / early_stopping apply to BEAM SEARCH only; passing
            # them with num_beams=1 triggers a warning on every call.
            kw["length_penalty"] = CFG["LENGTH_PENALTY"]
            kw["early_stopping"] = True
        return self.bart.generate(**kw)

model = CXRReportGenerator(S5_CKPT, CFG["DECODER"], CFG["UNFREEZE_VISION_STAGES"]).to(DEV)
tp = sum(p.numel() for p in model.parameters() if p.requires_grad)
fp = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f"  trainable {tp/1e6:.1f}M | frozen {fp/1e6:.1f}M")

# label smoothing fights the mode collapse that produced 14 unique openings /100
criterion_ls = nn.CrossEntropyLoss(ignore_index=-100, label_smoothing=CFG["LABEL_SMOOTH"])

def compute_loss(out, labels):
    return criterion_ls(out.logits.reshape(-1, out.logits.size(-1)).float(), labels.reshape(-1))

---
# 6 · Batch-size finder

In [ ]:
def probe(bs):
    try:
        torch.cuda.empty_cache(); gc.collect()
        x = torch.randn(bs, 3, CFG["IMG_SIZE"], CFG["IMG_SIZE"], device=DEV)
        y = torch.randint(0, 1000, (bs, CFG["MAX_TOKENS"]), device=DEV)
        opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=0.0)
        with torch.autocast("cuda", dtype=AMP_DT):
            loss = compute_loss(model(x, y), y)
        loss.backward(); opt.step(); opt.zero_grad(set_to_none=True)
        peak = torch.cuda.max_memory_allocated()/1e9
        del x, y, opt, loss
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); gc.collect()
        return True, peak
    except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
        if "out of memory" not in str(e).lower() and not isinstance(e, torch.cuda.OutOfMemoryError):
            raise
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); gc.collect()
        return False, 0.0

BATCH = None
for bs in (32, 24, 16, 12, 8, 4):
    ok, peak = probe(bs)
    print(f"  batch {bs:>3}: {'OK  ' if ok else 'OOM '}{f'peak {peak:.1f} GB' if ok else ''}")
    if ok: BATCH = bs; break
assert BATCH, "even batch 4 does not fit"
# probe used the worst case (every sequence padded to MAX_TOKENS); real batches are
# ~1/3 that length thanks to dynamic padding, so there is real headroom at runtime.
ACCUM = max(1, round(CFG["EFFECTIVE_BATCH"] / BATCH))
print(f"\n  batch={BATCH} × accum={ACCUM} → effective {BATCH*ACCUM}")
model.zero_grad(set_to_none=True)

---
# 7 · Loaders, optimiser, EMA

In [ ]:
LOADERS = {
    "train": DataLoader(DS["train"], batch_size=BATCH, shuffle=True, collate_fn=collate,
                        num_workers=CFG["NUM_WORKERS"], pin_memory=True, drop_last=True,
                        persistent_workers=True),
    "valsub": DataLoader(VAL_SUB, batch_size=BATCH, shuffle=False, collate_fn=collate,
                         num_workers=CFG["NUM_WORKERS"], pin_memory=True),
    "val":   DataLoader(DS["val"], batch_size=BATCH, shuffle=False, collate_fn=collate,
                        num_workers=CFG["NUM_WORKERS"], pin_memory=True),
    "test":  DataLoader(DS["test"], batch_size=BATCH, shuffle=False, collate_fn=collate,
                        num_workers=CFG["NUM_WORKERS"], pin_memory=True),
}
vision_params = [p for p in model.vision.parameters() if p.requires_grad]
proj_params   = list(model.proj.parameters())
bart_params   = list(model.bart.parameters())
LR = CFG["BASE_LR"] * (BATCH * ACCUM) / 32
groups = [{"params": bart_params, "lr": LR},
          {"params": proj_params, "lr": LR * CFG["PROJ_LR_MULT"]}]
if vision_params:
    groups.append({"params": vision_params, "lr": LR * CFG["VISION_LR_MULT"]})
optimizer = torch.optim.AdamW(groups, weight_decay=CFG["WEIGHT_DECAY"])

steps_per_epoch = max(1, len(LOADERS["train"]) // ACCUM)
warmup = CFG["WARMUP_EPOCHS"] * steps_per_epoch
total  = CFG["EPOCHS"] * steps_per_epoch
def lr_at(s):
    if s < warmup: return s / max(warmup, 1)
    p = (s - warmup) / max(total - warmup, 1)
    return max(0.01, 0.5 * (1 + math.cos(math.pi * p)))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_at)
scaler = torch.amp.GradScaler("cuda", enabled=(CFG["AMP_DTYPE"] == "fp16"))

class EMA:
    """Swap-based EMA — never allocates a second model on the GPU (Stage-5 fix)."""
    def __init__(self, model, decay):
        self.decay = decay
        self.shadow = {k: v.detach().clone().float() for k, v in model.state_dict().items()
                       if v.dtype.is_floating_point}
        self._backup = None
    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if k in self.shadow:
                self.shadow[k].mul_(self.decay).add_(v.detach().float(), alpha=1 - self.decay)
    @torch.no_grad()
    def apply(self, model):
        sd = model.state_dict()
        self._backup = {k: sd[k].detach().to("cpu", copy=True) for k in self.shadow}
        for k, v in self.shadow.items(): sd[k].copy_(v)
    @torch.no_grad()
    def restore(self, model):
        if self._backup is None: return
        sd = model.state_dict()
        for k, v in self._backup.items(): sd[k].copy_(v.to(sd[k].device))
        self._backup = None

# Stage-5 used a fixed 0.9998 → a 5,000-step window on a 22,000-step run, so 86%
# of the evaluated weights were still the random init at epoch 1. Derive it instead:
# aim for an averaging window of ~5% of total optimiser steps.
EMA_DECAY = float(np.clip(1 - 1/max(1.0, 0.05 * total), 0.99, 0.9999))
ema = EMA(model, EMA_DECAY) if CFG["USE_EMA"] else None
print(f"  LR bart={LR:.2e} proj={LR*CFG['PROJ_LR_MULT']:.2e}"
      + (f" vision={LR*CFG['VISION_LR_MULT']:.2e}" if vision_params else ""))
print(f"  {steps_per_epoch} steps/epoch | warmup {warmup} | total {total}")
print(f"  EMA decay {EMA_DECAY:.5f} → window ≈{1/(1-EMA_DECAY):.0f} steps "
      f"({1/(1-EMA_DECAY)/max(total,1)*100:.0f}% of training)")

---
# 8 · Metrics

ROUGE alone cannot see this problem — a constant string scores 0.2769. So we also
measure **mode collapse**, **prior-study hallucination**, and a lightweight
**clinical-efficacy F1** built from the Stage-3 evidence extractor.

In [ ]:
_SC = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

PRIOR_RE = re.compile(
    r"(as compared (to|with)|in compar(ison|ed) (to|with)"
    r"|\b(prior|previous|earlier|preceding)\s+(studies|study|exams?|radiographs?|films?|imaging)"
    r"|\b(unchanged|stable|constant)\b|\bno (significant |relevant |interval )?changes?\b"
    r"|\b(increasing|decreasing|worsening|improving)\b|___"
    r"|\b(again|persistent|persists|remains?)\b)", re.I)

PATH_KW = {
 "Cardiomegaly": r"(cardiomegaly|cardiac enlargement|enlarged cardiac silhouette|heart.{0,20}enlarged)",
 "Edema": r"(pulmonary edema|interstitial edema|\bedema\b|vascular congestion)",
 "Pleural_Effusion": r"(pleural effusion|\beffusions?\b)",
 "Atelectasis": r"atelecta", "Consolidation": r"consolidat",
 "Lung_Opacity": r"(opacit|infiltrate)", "Pneumonia": r"pneumonia",
 "Pneumothorax": r"pneumothora"}
NEG_RE = re.compile(r"\b(no|not|without|negative for|free of|absence of|absent)\b", re.I)

def assert_labels(text):
    """Positive assertion of each pathology, with sentence-scoped negation."""
    out = {}
    sents = re.split(r"(?<=[.;])\s+", re.sub(r"\s+", " ", text or ""))
    for lab, pat in PATH_KW.items():
        kw = re.compile(pat, re.I); pos = 0
        for s in sents:
            for m in kw.finditer(s):
                if not NEG_RE.search(s[:m.start()]): pos = 1; break
            if pos: break
        out[lab] = pos
    return out

def clinical_f1(preds, refs):
    """Micro-F1 over pathology assertions — does the report name the same findings?"""
    tp = fp = fn = 0
    for p, r in zip(preds, refs):
        a, b = assert_labels(p), assert_labels(r)
        for k in PATH_KW:
            if a[k] == 1 and b[k] == 1: tp += 1
            elif a[k] == 1 and b[k] == 0: fp += 1
            elif a[k] == 0 and b[k] == 1: fn += 1
    prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)
    return (2 * prec * rec / max(prec + rec, 1e-9)), prec, rec

def report_metrics(preds, refs):
    r1 = r2 = rl = 0.0
    for p, r in zip(preds, refs):
        s = _SC.score(r, p)
        r1 += s["rouge1"].fmeasure; r2 += s["rouge2"].fmeasure; rl += s["rougeL"].fmeasure
    n = max(len(preds), 1)
    firsts = [re.split(r"(?<=[.])\s", p.strip())[0] for p in preds if p.strip()]
    vp = {w for p in preds for w in p.lower().split()}
    vr = {w for r in refs for w in r.lower().split()}
    f1, prec, rec = clinical_f1(preds, refs)
    return dict(rouge1=r1/n, rouge2=r2/n, rougeL=rl/n,
                margin=rl/n - CONST_BASELINE_ROUGEL,
                uniq_reports=len(set(preds))/n,
                uniq_firsts=len(set(firsts))/max(len(firsts), 1),
                vocab_ratio=len(vp)/max(len(vr), 1),
                prior_rate=sum(bool(PRIOR_RE.search(p)) for p in preds)/n,
                gen_words=float(np.mean([len(p.split()) for p in preds])) if preds else 0.0,
                ref_words=float(np.mean([len(r.split()) for r in refs])) if refs else 0.0,
                clinical_f1=f1, clinical_prec=prec, clinical_rec=rec)

---
# 9 · Train / evaluate

In [ ]:
from tqdm.auto import tqdm
CKPT_EVERY_MIN = 10

def train_epoch(epoch, max_steps=None, ckpt_cb=None):
    global CKPT_EVERY_MIN
    model.train()
    if not model.unfrozen: model.vision.eval()
    tot = n = 0; t0 = last = time.time()
    optimizer.zero_grad(set_to_none=True)
    nb = max_steps or len(LOADERS["train"])
    pbar = tqdm(LOADERS["train"], total=nb, desc=f"  epoch {epoch:02d}", unit="batch",
                dynamic_ncols=True, leave=True)
    for i, (x, y, _) in enumerate(pbar):
        if max_steps and i >= max_steps: break
        x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
        with torch.autocast("cuda", dtype=AMP_DT):
            loss = compute_loss(model(x, y), y) / ACCUM
        if not torch.isfinite(loss):
            if ckpt_cb:
                try: ckpt_cb()
                except Exception: pass
            raise RuntimeError(f"non-finite loss at epoch {epoch} step {i}; lower BASE_LR and resume")
        (scaler.scale(loss).backward() if scaler.is_enabled() else loss.backward())
        if (i + 1) % ACCUM == 0:
            if scaler.is_enabled(): scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], CFG["GRAD_CLIP"])
            if scaler.is_enabled(): scaler.step(optimizer); scaler.update()
            else: optimizer.step()
            optimizer.zero_grad(set_to_none=True); scheduler.step()
            if ema: ema.update(model)
        tot += loss.item() * ACCUM; n += 1
        pbar.set_postfix_str(f"loss={tot/max(n,1):.4f} lr={optimizer.param_groups[0]['lr']:.1e} "
                             f"{(i+1)*BATCH/(time.time()-t0):.0f} img/s", refresh=False)
        if ckpt_cb and (time.time() - last) > CKPT_EVERY_MIN * 60:
            pbar.write(f"    [safety save @ step {i}/{nb}]"); ckpt_cb(); last = time.time()
    pbar.close()
    return tot / max(n, 1)

@torch.no_grad()
def evaluate(loader, beams, max_batches=None, show=0):
    model.eval()
    preds, refs, tl, nb_ = [], [], 0.0, 0
    for i, (x, y, texts) in enumerate(tqdm(loader, desc="  eval", leave=False,
                                           total=max_batches or len(loader))):
        if max_batches and i >= max_batches: break
        x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
        with torch.autocast("cuda", dtype=AMP_DT):
            tl += compute_loss(model(x, y), y).item(); nb_ += 1
            ids = model.generate(x, beams, CFG["GEN_MAX_TOKENS"], CFG["GEN_MIN_TOKENS"])
        preds += TOK.batch_decode(ids, skip_special_tokens=True)
        refs += list(texts)
    m = report_metrics(preds, refs); m["loss"] = tl / max(nb_, 1)
    for j in range(min(show, len(preds))):
        print(f"\n    REF: {' '.join(refs[j].split())[:200]}")
        print(f"    GEN: {' '.join(preds[j].split())[:200]}")
    return m, preds, refs

---
# 10 · Checkpointing

In [ ]:
BEST = CKPT_DIR / "best.pt"
LAST = CKPT_DIR / "last.pt"
LOCAL_LAST = Path("/content/stage4_last_local.pt")   # mid-epoch → local disk

def save_ckpt(path, epoch, best_metric, history, light=False):
    st = {"epoch": epoch, "best_metric": best_metric, "history": history, "cfg": CFG,
          "decoder": CFG["DECODER"], "batch": BATCH, "accum": ACCUM,
          "model": model.state_dict(), "ema": ema.shadow if ema else None}
    if not light:
        st |= {"optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
               "scaler": scaler.state_dict()}
    tmp = path.with_suffix(".tmp"); torch.save(st, tmp); tmp.replace(path)

def load_ckpt(path):
    c = torch.load(path, map_location="cpu", weights_only=False)
    model.load_state_dict(c["model"])
    for k, o in (("optimizer", optimizer), ("scheduler", scheduler), ("scaler", scaler)):
        if k in c: o.load_state_dict(c[k])
    if ema and c.get("ema"): ema.shadow = {k: v.to(DEV) for k, v in c["ema"].items()}
    return c["epoch"], c["best_metric"], c.get("history", [])

start_epoch, best_metric, history = 0, -1e9, []
cands = [p for p in (LOCAL_LAST, LAST) if p.exists()]
if CFG["RESUME"] and cands and not CFG["SMOKE_TEST"]:
    src = max(cands, key=lambda p: p.stat().st_mtime)
    start_epoch, best_metric, history = load_ckpt(src)
    print(f"  ▶ resumed from {src.name}: epoch {start_epoch}, best {best_metric:.4f}")
else:
    print("  starting fresh")

---
# 11 · SMOKE TEST

~6 min, ≈0.2 CU. Exercises every path and prints a real cost estimate.

In [ ]:
if CFG["SMOKE_TEST"]:
    print("=" * 80); print("  SMOKE TEST"); print("=" * 80)
    t0 = time.time()
    hits = []; _s = CKPT_EVERY_MIN; CKPT_EVERY_MIN = 0
    l = train_epoch(0, max_steps=30, ckpt_cb=lambda: hits.append(1))
    CKPT_EVERY_MIN = _s
    assert hits, "safety-save hook never fired"
    print(f"  train loop OK — loss {l:.4f} | safety hook fired {len(hits)}x")
    m, preds, refs = evaluate(LOADERS["valsub"], beams=1, max_batches=4, show=1)
    print(f"\n  eval OK — ROUGE-L {m['rougeL']:.4f} | uniq firsts {m['uniq_firsts']:.2f} "
          f"| prior-rate {m['prior_rate']:.2f} | clinical-F1 {m['clinical_f1']:.3f}")
    mb, _, _ = evaluate(LOADERS["valsub"], beams=CFG["TEST_BEAMS"], max_batches=2)
    print(f"  beam search OK — ROUGE-L {mb['rougeL']:.4f}")
    save_ckpt(LOCAL_LAST, 0, m["rougeL"], []); print(f"  checkpoint OK — {LOCAL_LAST.stat().st_size/1e6:.0f} MB")
    e0, b0, _ = load_ckpt(LOCAL_LAST); print(f"  resume OK — epoch {e0}")
    if ema:
        w0 = model.proj[1].weight.detach().clone()
        ema.apply(model); assert not torch.equal(model.proj[1].weight, w0)
        ema.restore(model); assert torch.equal(model.proj[1].weight, w0)
        print("  EMA apply/restore OK (no second GPU model)")
    ips = 30 * BATCH / (time.time() - t0)
    eta = len(DS["train"]) / max(ips, 1) * CFG["EPOCHS"] / 3600
    print(f"\n  throughput ≈ {ips:.0f} img/s")
    print(f"  ESTIMATED FULL RUN: {eta:.1f} h ≈ {eta*RATE:.0f} compute units")
    print("=" * 80); print("  ✅ SMOKE TEST PASSED"); print("=" * 80)
    print("\n  NEXT: set SMOKE_TEST = False, Runtime → Restart session, Run all.")
    raise SystemExit("smoke test complete — flip SMOKE_TEST to False")

---
# 12 · Training

Checkpoint selection is on **ROUGE-L over the fixed val subset**, so every epoch is
compared on identical images.

In [ ]:
print("=" * 80); print(f"  TRAINING  epochs {start_epoch+1} → {CFG['EPOCHS']}"); print("=" * 80)
patience = 0
for epoch in range(start_epoch + 1, CFG["EPOCHS"] + 1):
    te = time.time()
    tl = train_epoch(epoch, ckpt_cb=lambda: save_ckpt(LOCAL_LAST, epoch - 1, best_metric, history))
    if ema: ema.apply(model)
    m, preds, refs = evaluate(LOADERS["valsub"], beams=CFG["VAL_BEAMS"], show=(2 if epoch % 3 == 0 else 0))
    if ema: ema.restore(model)
    dt = time.time() - te; el = (time.time() - T_START) / 3600
    history.append({"epoch": epoch, "train_loss": tl, **{k: float(v) for k, v in m.items()}})
    star = ""
    if m["rougeL"] > best_metric:
        best_metric = m["rougeL"]; patience = 0; star = "  ** BEST"
        save_ckpt(BEST, epoch, best_metric, history, light=True)
    else:
        patience += 1
    save_ckpt(LAST, epoch, best_metric, history)
    print(f"\n  E{epoch:02d} loss={tl:.4f} vl={m['loss']:.4f} | ROUGE-L={m['rougeL']:.4f} "
          f"(margin {m['margin']:+.4f}) | clinF1={m['clinical_f1']:.3f} "
          f"| firsts={m['uniq_firsts']:.2f} prior={m['prior_rate']:.2f} "
          f"| {dt/60:.1f}min | {el:.1f}h ≈ {el*RATE:.0f} CU{star}", flush=True)
    if patience >= CFG["PATIENCE"]:
        print(f"\n  early stopping — no improvement for {CFG['PATIENCE']} epochs"); break

print(f"\n  DONE | best val ROUGE-L {best_metric:.4f} | "
      f"{(time.time()-T_START)/3600:.1f} h ≈ {(time.time()-T_START)/3600*RATE:.0f} CU")

---
# 13 · Test evaluation

Beam search on the **full** test set, against the constant-baseline control.

In [ ]:
if not BEST.exists():
    raise FileNotFoundError(f"{BEST} missing — no epoch improved. Check the log above.")
c = torch.load(BEST, map_location="cpu", weights_only=False)
model.load_state_dict(c["model"]); model.to(DEV)
if ema and c.get("ema"):
    ema.shadow = {k: v.to(DEV) for k, v in c["ema"].items()}; ema.apply(model)
print(f"  loaded best checkpoint (epoch {c['epoch']})")

tm, tpreds, trefs = evaluate(LOADERS["test"], beams=CFG["TEST_BEAMS"], show=3)

print("\n" + "=" * 88); print("  TEST RESULTS"); print("=" * 88)
print(f"  ROUGE-1 {tm['rouge1']:.4f} | ROUGE-2 {tm['rouge2']:.4f} | ROUGE-L {tm['rougeL']:.4f}")
print(f"\n  {'metric':<34}{'OLD model':>12}{'NEW':>12}{'target':>12}")
print("  " + "-" * 70)
rows = [("ROUGE-L", 0.2739, tm["rougeL"], "> 0.2769"),
        ("margin over constant baseline", -0.0030, tm["margin"], "> +0.02"),
        ("unique first sentences", 0.14, tm["uniq_firsts"], "> 0.50"),
        ("unique reports", 0.55, tm["uniq_reports"], "> 0.80"),
        ("vocabulary ratio", 0.24, tm["vocab_ratio"], "> 0.55"),
        ("prior-study hallucination", 0.63, tm["prior_rate"], "< 0.05"),
        ("clinical-efficacy F1", float("nan"), tm["clinical_f1"], "> 0.30"),
        ("mean generated words", 32.0, tm["gen_words"], f"~{tm['ref_words']:.0f}")]
for k, o, n_, t in rows:
    os_ = "  n/a" if o != o else f"{o:>12.4f}"
    print(f"  {k:<34}{os_}{n_:>12.4f}{t:>12}")
print("  " + "-" * 70)
print(f"\n  CONSTANT BASELINE (no image)  ROUGE-L = {CONST_BASELINE_ROUGEL:.4f}")
print(f"  THIS MODEL                    ROUGE-L = {tm['rougeL']:.4f}")
print(f"  MARGIN                                = {tm['margin']:+.4f}"
      f"   {'✅ learned from the image' if tm['margin'] > 0.01 else '⚠️ at or below baseline'}")

---
# 14 · Save artifacts

In [ ]:
res = {"stage": 4, "timestamp": datetime.now().isoformat(), "gpu": GPU,
       "decoder": CFG["DECODER"], "best_epoch": int(c["epoch"]),
       "best_val_rougeL": float(best_metric),
       "test": {k: float(v) for k, v in tm.items()},
       "constant_baseline_rougeL": CONST_BASELINE_ROUGEL,
       "config": CFG, "batch": BATCH, "accum": ACCUM,
       "hours": round((time.time()-T_START)/3600, 2),
       "compute_units_est": round((time.time()-T_START)/3600*RATE, 1),
       "history": history}
(REPORTS / "stage4_results.json").write_text(json.dumps(res, indent=2, default=str), encoding="utf-8")
with open(REPORTS / "stage4_samples.txt", "w", encoding="utf-8") as f:
    for i in range(min(100, len(tpreds))):
        f.write(f"--- [{i+1}]\n  REF: {' '.join(trefs[i].split())}\n  GEN: {' '.join(tpreds[i].split())}\n\n")
print(f"  ✅ {REPORTS/'stage4_results.json'}")
print(f"  ✅ {REPORTS/'stage4_samples.txt'}  (100 pairs — read these)")
print(f"  ✅ {BEST}")
print(f"\n  total {(time.time()-T_START)/3600:.2f} h ≈ {(time.time()-T_START)/3600*RATE:.0f} CU")
print("\n  ⚠️  Runtime → Manage sessions → terminate. Units burn while connected.")